# Семинар 6: Механизм внимания и Transformer для аудио

На этом занятии мы перейдём от RNN и CNN к архитектуре **Transformer Encoder**. Разберём:
1. Математику и реализацию `Scaled Dot-Product Attention` с нуля.
2. Многоголовое внимание (`Multi-Head Attention`).
3. Позиционное кодирование (Sinusoidal Positional Encoding).
4. Полный блок Encoder и модель `AudioTransformer`.
5. Обучение на задаче идентификации дикторов

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Фиксация случайности
torch.manual_seed(42)
np.random.seed(42)

# Выбор устройства
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

## 1. Подготовка данных (VCTK)

Мы используем подмножество датасета VCTK (идентификация диктора). Аудио приводится к фиксированной длине, извлекаются MFCC-признаки, которые подаются на вход последовательностной модели.

In [ ]:
# %pip install gdown -q

In [ ]:
import gdown
import zipfile

FILE_ID = "1LHeFn_uLpgjsyVKvwhmtLxZ6lhaVnw7u"
output = "VCTK.zip"

if not os.path.exists(output):
    gdown.download(id=FILE_ID, output=output, quiet=False)

# Распаковка
with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall(".")

In [ ]:
# Параметры аудио
SAMPLE_RATE = 16000
TARGET_SEC = 3.0
TARGET_LEN = int(SAMPLE_RATE * TARGET_SEC)
N_MFCC = 13
N_FFT = 1024
HOP_LENGTH = 512
WIN_LENGTH = 1024
N_MELS = 128

In [ ]:
from dataset import VCTKDataset

In [ ]:
# Укажите путь к распакованному датасету
DATASET_PATH = "./VCTK"
full_dataset = VCTKDataset(DATASET_PATH, TARGET_LEN)

train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size],
                                                 generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

NUM_CLASSES = len(full_dataset.id2label)

In [ ]:
mfcc = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "win_length": WIN_LENGTH,
        "n_mels": N_MELS,
        "center": False},
).to(DEVICE)

## 2. Scaled Dot-Product Attention

Классическое внимание вычисляет веса важности каждого элемента последовательности относительно остальных:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V $$

Масштабирование на $\sqrt{d_k}$ предотвращает насыщение softmax при больших значениях скалярного произведения.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value,):
        """
        query, key, value: (batch, heads, seq_len, d_k)
        """
        d_k = query.size(-1)

        # 1. Скалярное произведение и масштабирование
        scores = None

        # 2. Softmax по последнему измерению
        attn_weights = None
        attn_weights = self.dropout(attn_weights)

        # 3. Взвешенная сумма значений
        output = None
        return output, attn_weights

## 3. Multi-Head Attention (MHA)

MHA параллельно вычисляет $h$ независимых голов внимания, каждая в своём подпространстве:

$$ \text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O $$

Это позволяет модели фокусироваться на разных аспектах последовательности одновременно.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = None
        self.W_k = None
        self.W_v = None
        self.W_o = None

        self.attention = ScaledDotProductAttention(dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Линейные проекции
        Q = None
        K = None
        V = None

        # 2. Разбиение на головы: (batch, seq, d_model) -> (batch, heads, seq, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 3. Attention
        attn_output, attn_weights = self.attention(Q, K, V, mask)

        # 4. Склеивание голов: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 5. Выходная проекция
        output = None
        return output, attn_weights

## 4. Positional Encoding

Transformer не имеет встроенного механизма порядка. Добавим к эмбеддингам синусоидальные позиционные кодировки:

$$ PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) $$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = None

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## 5. Transformer Encoder Block

Блок состоит из:
1. **Multi-Head Self-Attention**
2. **Add & LayerNorm**
3. **Feed-Forward Network** (Linear → GELU → Linear)
4. **Add & LayerNorm**

$$ \text{Output} = \text{LayerNorm}(x + \text{FFN}(\text{LayerNorm}(x + \text{MHA}(x)))) $$

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = None
        self.norm1 = None
        self.ffn = nn.Sequential()
        self.norm2 = None
        self.dropout = None

    def forward(self, x):
        # Attention + Residual + Norm


        # FFN + Residual + Norm


        return x

## 6. Полная модель AudioTransformer

Архитектура:
1. Линейный проектор `(n_mfcc → d_model)`
2. Позиционное кодирование
3. Стеки Encoder блоков
4. Global Mean Pooling по времени
5. Классификационная голова

In [ ]:
class AudioTransformer(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, num_layers, d_ff, num_classes, max_len=1000, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model)

        self.pool = nn.AdaptiveAvgPool1d(1)  # (batch, d_model, seq) -> (batch, d_model, 1)
        self.head = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        # x: (batch, seq_len, n_mfcc)
        x = self.input_proj(x)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_out(x)

        # Pooling: (batch, seq, d_model) -> (batch, d_model, 1)
        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)

        logits = self.head(self.dropout(x))

        return logits

# Инициализация
D_MODEL = 64
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 128

model = AudioTransformer(
    input_dim=N_MFCC, d_model=D_MODEL, num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS, d_ff=D_FF, num_classes=NUM_CLASSES, dropout=0.1
).to(DEVICE)
print(model)

In [ ]:
from torchinfo import summary

summary(model, input_size=(10, 100, 13))

In [ ]:
model = model.to(DEVICE)

## 7. Цикл обучения

Стандартный пайплайн: `AdamW` оптимизатор, `StepLR` шедулер, `CrossEntropyLoss`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for waveforms, labels in tqdm(loader, desc="Train", leave=False):
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            feats = mfcc(waveforms).transpose(1, 2)  # (batch, seq, mfcc)

        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for waveforms, labels in tqdm(loader, desc="Eval", leave=False):
        waveforms, labels = waveforms.to(DEVICE), labels.to(DEVICE)
        feats = mfcc(waveforms).transpose(1, 2)
        logits = model(feats)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

## 8. Запуск обучения

In [ ]:
EPOCHS = 5
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1:02d} | LR: {scheduler.get_last_lr()[0]:.5f} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2%} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2%}")

## 9. Визуализация обучения

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], marker="o", label="Train")
plt.plot(history["val_loss"], marker="s", label="Val")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.grid(alpha=0.3); plt.legend(); plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], marker="o", label="Train")
plt.plot(history["val_acc"], marker="s", label="Val")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.grid(alpha=0.3); plt.legend(); plt.title("Accuracy")
plt.tight_layout()
plt.show()